# FingerBall — Stage 2I-1: frozen 2D hand kinematics and QC

手動アノテーションを**変更せず**、関節角度・可視性・骨長の時系列を解析します。推定点と手動点の由来を保持し、疑似3Dを実測3Dとして扱いません。MANOメッシュフィッティングはこのNotebookの対象外です。

In [ ]:
from pathlib import Path
import json,numpy as np,pandas as pd,matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

## 1. 入力：Stage 2Hの確定データ

`INPUT_CSV` に Google Drive の `hand_21_landmarks_2d_corrected.csv` を指定してください。出力は別ディレクトリへ保存します。元CSVは一切変更しません。

In [ ]:
INPUT_CSV='/content/drive/MyDrive/母指回内トラッキング/FingerBall_v2_stage2h_results/7A5812501DC5F7120DF842D06EE1B74C695459F1/hand_21_landmarks_2d_corrected.csv'
OUTPUT_DIR='/content/drive/MyDrive/母指回内トラッキング/FingerBall_v2_stage2i1_results'
RELEASE_FRAME=190
F0,F1=185,198
path=Path(INPUT_CSV);out=Path(OUTPUT_DIR);out.mkdir(parents=True,exist_ok=True)
if not path.is_file():raise FileNotFoundError(f'Stage 2Hの確定CSVを入力してください: {path}')
df=pd.read_csv(path)
required={'frame','landmark_id','x_px','y_px','annotation_status','xy_source'}
assert required.issubset(df.columns),required-set(df.columns)
assert not df.duplicated(['frame','landmark_id']).any(),'Duplicate landmark rows'
subset=df[df.frame.between(F0,F1)].copy()
assert len(subset)==(F1-F0+1)*21, 'Expected 21 landmarks per frame'
assert subset.groupby('frame').landmark_id.nunique().eq(21).all()
print(subset.annotation_status.value_counts())

## 2. 2D角度と指先間距離（可視点だけ）

見えない点はNaNを維持します。**手動確定か候補かの出所を各指標に付与**し、今後の感度分析に使用します。関節角度は2D投影角度であり、真の屈伸角ではありません。

In [ ]:
specs={
 'thumb_IP_2D':(2,3,4),
 'index_PIP_2D':(5,6,7),'index_DIP_2D':(6,7,8),
 'middle_PIP_2D':(9,10,11),'middle_DIP_2D':(10,11,12),
 'ring_PIP_2D':(13,14,15),'little_PIP_2D':(17,18,19)}
levels={'manual_visible':0,'ai':1,'propagated_suggestion':2,'interpolated':3,'occluded':4,'missing':4}
def angle(p,a,b,c):
    if not np.isfinite(p[[a,b,c]]).all():return np.nan
    u=p[a]-p[b];v=p[c]-p[b]
    if min(np.linalg.norm(u),np.linalg.norm(v))<1e-8:return np.nan
    return float(np.degrees(np.arccos(np.clip(np.dot(u,v)/(np.linalg.norm(u)*np.linalg.norm(v)),-1,1))))
rows=[]
for f,g in subset.groupby('frame'):
    g=g.set_index('landmark_id').reindex(range(21))
    p=g[['x_px','y_px']].to_numpy(float)
    statuses=g.annotation_status.fillna('missing').to_dict()
    row={'frame':int(f),'time_s':float(g.time_s.iloc[0]) if 'time_s' in g else np.nan}
    for key,(a,b,c) in specs.items():
        row[key]=angle(p,a,b,c)
        involved=[statuses[x] for x in (a,b,c)]
        row[key+'_worst_status']=max(involved,key=lambda x:levels.get(x,5))
        row[key+'_all_manual']=all(x=='manual_visible' for x in involved)
    if np.isfinite(p[[8,12]]).all():row['index_middle_tip_distance_px']=float(np.linalg.norm(p[8]-p[12]))
    else:row['index_middle_tip_distance_px']=np.nan
    rows.append(row)
kin=pd.DataFrame(rows)
kin.to_csv(out/'stage2i1_projected_kinematics_with_provenance.csv',index=False)
print(kin.round(2).to_string(index=False))

## 3. 時系列表示と骨長QC

In [ ]:
fig,ax=plt.subplots(figsize=(11,5))
for col in ['index_PIP_2D','index_DIP_2D','middle_PIP_2D','middle_DIP_2D']:
    ax.plot(kin.frame,kin[col],marker='o',label=col)
ax.axvline(RELEASE_FRAME,color='black',linestyle='--',alpha=.6,label='release reference')
ax.set(xlabel='Original frame',ylabel='2D projected included angle (degrees)',title='FingerBall 2D hand kinematics: provenance retained')
ax.legend(ncol=2);ax.grid(alpha=.25);fig.tight_layout()
fig.savefig(out/'stage2i1_projected_angles.png',dpi=160)
plt.show()
segments={'index_MCP_PIP':(5,6),'index_PIP_DIP':(6,7),'index_DIP_TIP':(7,8),
          'middle_MCP_PIP':(9,10),'middle_PIP_DIP':(10,11),'middle_DIP_TIP':(11,12)}
length_rows=[]
for f,g in subset.groupby('frame'):
    g=g.set_index('landmark_id').reindex(range(21))
    p=g[['x_px','y_px']].to_numpy(float)
    row={'frame':int(f)}
    for name,(i,j) in segments.items():
        row[name+'_px']=float(np.linalg.norm(p[i]-p[j])) if np.isfinite(p[[i,j]]).all() else np.nan
    length_rows.append(row)
lengths=pd.DataFrame(length_rows)
lengths.to_csv(out/'stage2i1_bone_length_qc.csv',index=False)
print('Saved to',out)

## 次のStage 2I-2

この2D計測と修正済み元座標を使用し、MANOなどの3D手モデルの2D再投影誤差を最小化するフィッティングへ進めます。単眼3Dは不定性があるため、モデル依存の推定結果として扱い、修正済みXYを上書きしません。遮蔽点は拘束対象から除外し、独立の2D評価フレームで再投影精度を検証します。